In [2]:
import numpy as np
import copy
import subprocess
from pathlib import Path
import matplotlib.pyplot as plt

In [3]:
# some functions
def relu(x):
	return np.maximum(x,0)

PARAMS = [
    "children_to_ctx",
    "W1", "b1",
    "W2", "b2",
    "W3"
]

In [ ]:
# bring randomly generated instructions
def bring_shit(len_program=300):
	result=subprocess.run(["./rand_gen","test.s","--num_instructions",f"{len_program}"]
						,text=True,
						capture_output=True
						)

	result=subprocess.run(["./build/kites-cli","test.s","--reorder","2","--vm","rv5-h-nf"]
						,text=True,
						capture_output=True
						)

	print(result.stderr)
	raw_data=np.fromfile("./ml/input_adj.bin",dtype=np.int32)
	num_instructions=int(raw_data[0])

	adjmatrix=raw_data[1:].reshape((num_instructions,num_instructions)).astype(np.float32) # weighted edges

	me=np.fromfile("./ml/input_me.bin",dtype=np.int32)
	me_vector = me.reshape((num_instructions,5)).astype(np.float32) # num_inst x 5 in og order

	return adjmatrix, me_vector

In [5]:
# the goat model
class Model:
	def __init__(self):
		# initial matrix
		self.children_to_ctx = np.random.rand(59, 65)

		# mlp matrices
		self.W1 = np.random.rand(128,64)
		self.b1 = np.random.rand(128)

		self.W2 = np.random.rand(32,128)
		self.b2 = np.random.rand(32)

		self.W3 = np.random.rand(1,32)

	def me_to_id_all(self, me, adj):
			# instructions_id_array = np.concatenate((instructions_me_array, np.zeros((len(instructions_me_array), 59))), axis=1)
			# # traverse from last
			# num = instructions_me_array.size
	
			# for i in range(num-1, -1, -1):
			# 	# check if not leaf
			# 	if np.any(adjmatrix[i]):
			# 		mask = adjmatrix[i]>0 # children
			# 		ctx = np.r_[np.sum(adjmatrix[i]), np.sum(instructions_id_array[mask], axis=0)]
			# 		ctx = self.children_to_ctx @ ctx
			# 		instructions_id_array[i,5:] = ctx
			# return instructions_id_array

		n = len(me)
		ids = np.concatenate([me, np.zeros((n, 59))], axis=1)
		for i in range(n - 1, -1, -1):
			w = adj[i]; mask = w > 0
			if mask.any():
				agg = (w[mask] @ ids[mask]) / w[mask].sum()      # edge-weighted mean
				ids[i, 5:] = np.tanh(self.children_to_ctx @ np.r_[w[mask].sum(), agg])
		return ids

	def forward(self, instruction_id):
		# me + ctx
		a1 = self.W1 @ instruction_id + self.b1
		z1 = relu(a1)

		a2 = self.W2 @ z1 + self.b2
		z2 = relu(a2)

		priority = self.W3 @ z2
		return priority

	def priority_setter(self, instruction_id_array):
		priorities = np.array([self.forward(row) for row in instruction_id_array])
		return priorities
		

In [6]:
# would get the reward
def reward(model, instruction_me_vector, adjmatrix):
	instruction_id_vector = model.me_to_id_all(instruction_me_vector, adjmatrix)
	ps = model.priority_setter(instruction_id_vector)

	ps.tofile("./ml/priority_out.bin")

	result = subprocess.run(
		"./build/kites-cli -test.s --compare-reorder 3 --vm rv5-h-nf --cycles-original cycles1.txt --cycles-modified cycles2.txt".split(' '), text=True, capture_output=True
	)
	
	file1=Path("cycles1.txt")
	file2=Path("cycles2.txt")
	cycles_original=int(file1.read_text().strip())
	cycles_modified=int(file2.read_text().strip())
	file1.unlink()
	file2.unlink()

	result=(cycles_original-cycles_modified)/cycles_original

	return result

In [7]:
# for a model returns an array of all weights
def get_weights(model):
    return np.concatenate([getattr(model, name).flatten() for name in PARAMS])

# given weight array, sets the weights
def set_weights(model, weights):
    start = 0

    for name in PARAMS:
        old = getattr(model, name)
        size = old.size

        new = weights[start:start + size].reshape(old.shape)
        setattr(model, name, new.copy())

        start += size


def es_step(model, reward_fn, me_vector, adjmatrix, N=50, sigma=0.1, alpha=0.01):

    w = get_weights(model)
    half = np.random.randn(N // 2, w.size)
    noise = np.concatenate([half, -half])
    raw = np.empty(N)
    for i in range(N):
        cand = copy.deepcopy(model); set_weights(cand, w + sigma * noise[i])
        raw[i] = reward_fn(cand, me_vector, adjmatrix)
    ranks = np.empty(N); ranks[raw.argsort()] = np.arange(N)
    shaped = ranks / (N - 1) - 0.5
    set_weights(model, w + alpha * (shaped @ noise) / (N * sigma))
    return raw.mean()  

In [10]:
if __name__ == "__main__":
	CoRRL = Model()
	all_rewards = []

	how_many_assem = 10
	for i in range(how_many_assem):
		adjmatrix, me_vector = bring_shit(300)

		# curr_reward = reward(CoRRL, me_vector, adjmatrix)
		
		curr_reward = es_step(CoRRL, reward_fn=reward, me_vector=me_vector, adjmatrix=adjmatrix, N=10)
		all_rewards.append(curr_reward)

		print(f"Done with {i+1}/{how_many_assem} random generations. With current reward = {curr_reward}")

	plt.plot(all_rewards)
	plt.show()


FileNotFoundError: [Errno 2] No such file or directory: './ml/input_adj.bin'